# Week 8 Data Story: Analysis Notebook

**Title:** When the City Emptied, Crime Didn't Vanish: It Moved

**Author:** DTU 02806 Social Data Analysis & Visualization

This notebook documents the exploratory analysis, data preparation, and visualization creation process behind the Week 8 data story. The final published version is available at `index.html`.

## Purpose

This notebook serves three functions:
1. **Transparency**: Show the analytical decisions behind the published story
2. **Reproducibility**: Document the code used to generate all visualizations
3. **Critical reflection**: Discuss what worked, what didn't, and what got left out

---

## 1. Setup and Data Loading

We use the merged SF crime dataset (2003–2024) created in earlier weeks.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import plotly.express as px
import folium
from folium.plugins import HeatMap
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("✓ Libraries loaded")

In [ ]:
# Load the merged dataset
DATA_PATH = "../merged_crime_data_2003_2025.csv"

df = pd.read_csv(
    DATA_PATH,
    parse_dates=["Incident_Date"],
    dtype={"Police_District": "category", "Unified_Category": "category"},
)

print(f"Total rows: {len(df):,}")
print(f"Date range: {df['Incident_Date'].min()} to {df['Incident_Date'].max()}")
print(f"\nColumns: {df.columns.tolist()}")

## 2. Exploratory Data Analysis

### 2.1 Check Data Quality

In [ ]:
# Missing values
print("Missing values by column:")
print(df.isnull().sum())
print(f"\nPercentage of rows with missing lat/lon: {(df['Latitude'].isnull().sum() / len(df)) * 100:.2f}%")

In [ ]:
# Year distribution
print("Incidents by year:")
print(df['Year'].value_counts().sort_index())

# Filter to complete years only (2003-2024)
df_full = df[(df['Year'] >= 2003) & (df['Year'] <= 2024)].copy()
print(f"\nAfter filtering to complete years: {len(df_full):,} rows")

### 2.2 Focus Crime Categories

For the data story, I focused on five consistently coded categories that matter to residents and span the entire time period.

In [ ]:
# Check all available categories
print("All crime categories and counts:")
print(df_full['Unified_Category'].value_counts())

FOCUS_CRIMES = ['Assault', 'Burglary', 'Larceny/Theft', 'Robbery', 'Vehicle Theft']

df_focus = df_full[df_full['Unified_Category'].isin(FOCUS_CRIMES)].copy()
print(f"\nFocus crimes represent {len(df_focus):,} incidents ({(len(df_focus)/len(df_full))*100:.1f}% of total)")

### 2.3 District Name Standardization

The raw data has inconsistent capitalization. This needs cleaning before aggregation.

In [ ]:
# Check district names
print("Raw district names:")
print(df_full['Police_District'].value_counts())

DISTRICT_MAP = {
    "BAYVIEW": "Bayview",
    "CENTRAL": "Central",
    "INGLESIDE": "Ingleside",
    "MISSION": "Mission",
    "NORTHERN": "Northern",
    "PARK": "Park",
    "RICHMOND": "Richmond",
    "SOUTHERN": "Southern",
    "TARAVAL": "Taraval",
    "TENDERLOIN": "Tenderloin",
}

# Normalize
df_focus['Police_District_Clean'] = (
    df_focus['Police_District']
    .astype(str)
    .str.strip()
    .str.upper()
    .map(DISTRICT_MAP)
    .fillna("Unknown")
)

print("\nAfter standardization:")
print(df_focus['Police_District_Clean'].value_counts())

## 3. Key Statistics for the Story

### 3.1 Pre-COVID vs COVID-Era Comparison

These numbers appear in **Table 1** in the published story.

In [ ]:
# Define periods
BASELINE_YEAR = 2019
COVID_YEARS = [2020, 2021]

# Get 2019 baseline
df_2019 = df_focus[df_focus['Year'] == BASELINE_YEAR]
counts_2019 = df_2019.groupby('Unified_Category').size()

# Get 2020-21 average
df_covid = df_focus[df_focus['Year'].isin(COVID_YEARS)]
counts_covid = df_covid.groupby('Unified_Category').size() / len(COVID_YEARS)

# Calculate percentage changes
comparison = pd.DataFrame({
    '2019_Count': counts_2019,
    '2020-21_Avg': counts_covid.astype(int),
    'Change_%': ((counts_covid - counts_2019) / counts_2019 * 100).round(1)
}).sort_values('Change_%', ascending=False)

print("Table 1 Data: 2019 vs 2020-21")
print(comparison)

print("\nKey takeaways:")
print(f"- Burglary increased by {comparison.loc['Burglary', 'Change_%']:+.1f}% (+{int(counts_covid['Burglary'] - counts_2019['Burglary']):,} incidents/year)")
print(f"- Vehicle Theft increased by {comparison.loc['Vehicle Theft', 'Change_%']:+.1f}% (+{int(counts_covid['Vehicle Theft'] - counts_2019['Vehicle Theft']):,} incidents/year)")
print(f"- Larceny/Theft decreased by {comparison.loc['Larceny/Theft', 'Change_%']:.1f}% ({int(counts_covid['Larceny/Theft'] - counts_2019['Larceny/Theft']):,} incidents/year)")

## 4. Visualization Creation

This section documents the code and design decisions for each of the three visualizations.

### 4.1 Figure 1: Static Time Series (Matplotlib)

**Design decisions:**
- Line chart to show temporal trends
- COVID period (2020-21) highlighted with shaded region
- Each crime type gets a distinct color
- Y-axis uses comma separators for readability
- Saved as high-DPI PNG for web display

In [ ]:
# Prepare yearly counts
yearly_counts = df_focus.groupby(['Year', 'Unified_Category']).size().reset_index(name='Count')

# Crime colors (consistent across all visualizations)
CRIME_COLORS = {
    'Assault': '#e63946',
    'Burglary': '#f77f00',
    'Larceny/Theft': '#457b9d',
    'Robbery': '#2a9d8f',
    'Vehicle Theft': '#8338ec'
}

# Create figure
fig, ax = plt.subplots(figsize=(10, 5.5), facecolor='#fafafa')
ax.set_facecolor('#fafafa')

# Plot each crime type
for crime in FOCUS_CRIMES:
    subset = yearly_counts[yearly_counts['Unified_Category'] == crime]
    ax.plot(
        subset['Year'], subset['Count'],
        color=CRIME_COLORS[crime],
        linewidth=2.2,
        marker='o', markersize=3.5,
        label=crime
    )

# COVID shading
ax.axvspan(2020, 2021.99, color='#e63946', alpha=0.08)
ax.annotate(
    'COVID-19', xy=(2021, ax.get_ylim()[1] * 0.92),
    fontsize=9, color='#e63946', fontweight='bold', ha='center'
)

# Styling
ax.set_xlabel('Year', fontsize=11)
ax.set_ylabel('Number of Incidents', fontsize=11)
ax.set_title('Yearly Crime Trends in San Francisco (2003–2024)', fontsize=13, fontweight='bold', pad=12)
ax.legend(frameon=True, fontsize=9, loc='upper left', framealpha=0.9)
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.set_xlim(2003, 2024)
ax.grid(axis='y', color='#dee2e6', linewidth=0.6)
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig('../images/yearly_crime_trends.png', dpi=180, bbox_inches='tight', facecolor='#fafafa')
plt.show()

print("✓ Figure 1 saved to images/yearly_crime_trends.png")

### 4.2 Figure 2: Interactive Heatmap (Folium)

**Design decisions:**
- Two-layer heatmap: Pre-COVID (2017-19) vs During COVID (2020-21)
- Same sample size (15,000 points) for both layers to keep visual intensity comparable
- Different color gradients to distinguish layers
- Layer control allows toggling between periods
- Focus on burglary because it showed significant spatial redistribution

In [ ]:
# Define periods
PRE_COVID_YEARS = [2017, 2018, 2019]
COVID_YEARS = [2020, 2021]

# Get burglary points
def get_geo_points(years, crime_type='Burglary'):
    subset = df_focus[
        (df_focus['Year'].isin(years)) &
        (df_focus['Unified_Category'] == crime_type) &
        df_focus['Latitude'].notna() &
        df_focus['Longitude'].notna()
    ]
    return subset[['Latitude', 'Longitude']]

pre_points = get_geo_points(PRE_COVID_YEARS)
covid_points = get_geo_points(COVID_YEARS)

print(f"Pre-COVID burglaries with coordinates: {len(pre_points):,}")
print(f"COVID burglaries with coordinates: {len(covid_points):,}")

# Sample equally
sample_n = min(len(pre_points), len(covid_points), 15000)
pre_sample = pre_points.sample(sample_n, random_state=42)
covid_sample = covid_points.sample(sample_n, random_state=42)

# Create map
sf_center = [37.76, -122.44]
m = folium.Map(location=sf_center, zoom_start=12, tiles='cartodbpositron')

common_kwargs = {
    'radius': 10,
    'blur': 12,
    'max_zoom': 13,
    'min_opacity': 0.35
}

# Pre-COVID layer (blue gradient)
HeatMap(
    pre_sample.values.tolist(),
    name='Pre-COVID (2017–2019)',
    gradient={0.2: '#457b9d', 0.5: '#1d3557', 0.8: '#e63946', 1.0: '#e63946'},
    **common_kwargs
).add_to(m)

# COVID layer (red gradient, hidden by default)
covid_fg = folium.FeatureGroup(name='During COVID (2020–2021)', show=False)
HeatMap(
    covid_sample.values.tolist(),
    gradient={0.2: '#f4a261', 0.5: '#e76f51', 0.8: '#e63946', 1.0: '#9b2226'},
    **common_kwargs
).add_to(covid_fg)
covid_fg.add_to(m)

folium.LayerControl(collapsed=False).add_to(m)

m.save('../visualizations/burglary_heatmap.html')
print("✓ Figure 2 saved to visualizations/burglary_heatmap.html")

# Display in notebook
m

### 4.3 Figure 3: Interactive Grouped Bar Chart (Plotly)

**Design decisions:**
- Faceted bar chart: one panel per period (Pre/During/Post-COVID)
- Grouped bars allow district-by-district comparison within each period
- Annualized counts (incidents per year) to make 2-year and 3-year periods comparable
- Different from Week 6 Plotly work (which showed hourly distributions and animations)
- Legend allows toggling crime types for focused comparisons

In [ ]:
# Define periods
PRE_COVID = [2017, 2018, 2019]
DURING_COVID = [2020, 2021]
POST_COVID = [2022, 2023, 2024]

def label_period(year):
    if year in PRE_COVID:
        return 'Pre-COVID (2017–19)'
    elif year in DURING_COVID:
        return 'COVID (2020–21)'
    elif year in POST_COVID:
        return 'Post-COVID (2022–24)'
    return None

# Filter and label
df_periods = df_focus[df_focus['Year'].isin(PRE_COVID + DURING_COVID + POST_COVID)].copy()
df_periods['Period'] = df_periods['Year'].apply(label_period)

# Aggregate and annualize
period_lengths = {
    'Pre-COVID (2017–19)': 3,
    'COVID (2020–21)': 2,
    'Post-COVID (2022–24)': 3
}

counts = (
    df_periods
    .groupby(['Police_District_Clean', 'Period', 'Unified_Category'])
    .size()
    .reset_index(name='TotalCount')
)

counts['YearsInPeriod'] = counts['Period'].map(period_lengths)
counts['AvgYearlyCount'] = (counts['TotalCount'] / counts['YearsInPeriod']).round(0).astype(int)

# Keep only main SF districts
MAIN_DISTRICTS = ['Bayview', 'Central', 'Ingleside', 'Mission', 'Northern',
                  'Park', 'Richmond', 'Southern', 'Taraval', 'Tenderloin']

counts_clean = counts[counts['Police_District_Clean'].isin(MAIN_DISTRICTS)].copy()

# Create Plotly figure
fig = px.bar(
    counts_clean,
    x='Police_District_Clean',
    y='AvgYearlyCount',
    color='Unified_Category',
    facet_col='Period',
    barmode='group',
    color_discrete_map=CRIME_COLORS,
    labels={
        'AvgYearlyCount': 'Avg. Yearly Incidents',
        'Police_District_Clean': 'District',
        'Unified_Category': 'Crime Type'
    },
    title='How Crime Shifted Across SF Districts: Before, During, and After COVID',
    category_orders={
        'Period': ['Pre-COVID (2017–19)', 'COVID (2020–21)', 'Post-COVID (2022–24)'],
        'Police_District_Clean': MAIN_DISTRICTS
    }
)

fig.update_layout(
    font_family='Helvetica Neue, Helvetica, Arial, sans-serif',
    plot_bgcolor='#fafafa',
    paper_bgcolor='white',
    height=620,
    margin=dict(t=90, b=145, l=75, r=35),
    legend=dict(
        title='Crime Type',
        orientation='h',
        yanchor='top', y=-0.18,
        xanchor='center', x=0.5
    )
)

fig.update_xaxes(tickangle=-35, automargin=True, title_standoff=8)
fig.update_yaxes(automargin=True, title_standoff=12, tickformat=',')
fig.for_each_annotation(lambda a: a.update(text=a.text.split('=')[1]))

fig.write_html('../visualizations/district_comparison.html', include_plotlyjs='cdn')
print("✓ Figure 3 saved to visualizations/district_comparison.html")

fig.show()

## 5. Critical Reflection

### 5.1 What Worked

- **Clear narrative arc**: The three visualizations build a logical argument from citywide → spatial → district-level
- **Appropriate chart types**: Time series for trends, heatmap for spatial patterns, faceted bars for multi-dimensional comparison
- **Design consistency**: Shared color palette and typography across all figures
- **Honest limitations**: The conclusion explicitly addresses reporting bias and causation vs. correlation

### 5.2 What Got Left Out

**Alternative visualizations I considered but rejected:**

1. **Small multiples (one map per district)**: Would show spatial variation within districts, but 10 small maps would be cluttered
2. **Animated time series**: Week 6 already explored animation; static was clearer for this narrative
3. **Stacked area chart**: Would emphasize total volume changes, but obscures the divergence pattern
4. **Sankey diagram (crime flow)**: Conceptually appealing but crime doesn't literally "flow" between categories

**Data limitations:**

- No demographic data (can't analyze disparate impacts by race/income)
- No resolution data (can't distinguish cleared vs. unsolved cases)
- No victim characteristics (can't separate resident vs. visitor victimization)
- Geographic coordinates missing for ~8% of incidents

### 5.3 If I Had More Time

1. **Statistical testing**: Formal change-point detection for COVID effect
2. **Spatial statistics**: Getis-Ord Gi* to identify statistically significant hotspots
3. **Contextual data integration**: Overlay foot traffic data (Google mobility reports) to directly test opportunity theory
4. **Victim perspective**: Interview SF residents in Sunset/Richmond about burglary concerns
5. **Comparison city**: Analyze Oakland or San Jose to see if patterns are SF-specific or regional

---

## 6. Reproducibility

To regenerate all visualizations:

```bash
cd scripts/
python main.py
```

This runs the modular scripts:
- `static_chart.py` → Figure 1
- `map_viz.py` → Figure 2
- `interactive_viz.py` → Figure 3

All use shared configuration from `config.py` and data loading from `data_loader.py`.

---

## Conclusion

This notebook documents the analytical process behind "When the City Emptied, Crime Didn't Vanish: It Moved."

**Key analytical choices:**
1. Focus on 5 consistently coded crimes (not all 20+ categories)
2. Annualize counts to make periods with different lengths comparable
3. Standardize district names before aggregation
4. Use equal sample sizes in heatmap layers for fair visual comparison
5. Choose visualization types that directly support each part of the argument

**Main finding:** COVID-19 didn't uniformly increase or decrease SF crime—it redistributed risk across crime types (larceny → vehicle theft) and space (commercial corridors → residential neighborhoods). The story is heterogeneity, not uniformity.

**Ethical stance:** This analysis uses data responsibly by:
- Acknowledging reporting bias (Richardson et al.)
- Avoiding causal claims without evidence
- Contextualizing patterns with theory (routine activity, opportunity)
- Being transparent about what the data cannot show

---

*Notebook created: 2024*  
*DTU Course 02806: Social Data Analysis and Visualization*